# Notebook B (Hindi): entity annotation of one long-form Hindi dataset, with web search, built on the paper's own pipeline

This notebook labels the long-form Hindi answers in one `longfact-hi-<model>` dataset (from Notebook A), using the paper's annotation code
([obalcells/hallucination_probes](https://github.com/obalcells/hallucination_probes) `annotation_pipeline/`).
Run one copy per model to do both models in parallel, for example on two Kaggle accounts.

- **Prompt:** the repo's `entity_annotation.prompt` (the paper's prompt), filled in by the repo's `format_prompt`.
  The instruction is the Hindi question and the completion is the Hindi answer.
  With `HINDI_NOTE = True`, one short note is added at the end: copy spans exactly in Devanagari (no translating or
  transliterating), search in English or Hindi, and write the notes in English. That is the only change to the paper's prompt.
- **Parsing and span positions:** the repo's `parse_and_validate_json`, `AnnotatedSpan` and `assign_span_positions`.
  The repo's fuzzy span matcher gets a Unicode tokenizer, because ROUGE's default tokenizer deletes Devanagari.
- **Judge:** gpt-oss-120b on HF credits at temperature 0, with up to 8,192 output tokens and up to 15 `web_search` calls per answer (repo `run.py`).
  Searches go to Exa's web-search MCP server; when Exa rate-limits, they go to DuckDuckGo.
- **Sample:** `FILES_LIMIT = 3` takes Notebook A's first 3 files, which are the same 60 prompts for Aya and Llama (paired).

**Saving:** every finished answer is written to `work/<dataset>/progress.jsonl` at once, and that file is uploaded to Hugging Face every 5 answers.
Each finished file of 20 answers is uploaded too. If the session stops, Run all again: finished answers are skipped.

**Kaggle:** Internet **on**, no GPU needed. Add-ons → Secrets → **`HF_TOKEN`**: the shreshtha2002 token (write + "Make calls to Inference Providers").

In [ ]:
import sys, subprocess, importlib, site, io, os, urllib.request, zipfile
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "nltk", "absl-py", "ddgs", "openai",
                "huggingface_hub", "pydantic>=2.6"], check=True)
site.addsitedir(site.getusersitepackages()); importlib.invalidate_caches()
import rouge_score, ddgs, openai
if not os.path.isdir("repo"):      # the paper's repo (no git needed)
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(
        "https://github.com/obalcells/hallucination_probes/archive/refs/heads/main.zip").read())).extractall(".")
    os.rename("hallucination_probes-main", "repo")
print("ready")

In [ ]:
# ---------------- settings ----------------
MODEL_REPO = "longfact-hi-llama3.1-8b-instruct-q8-0"   # the one dataset this notebook annotates (from Notebook A)
SRC_USER = "shreshthamodi02"            # account that holds the Hindi answers (public, read only)
HF_USER = "shreshtha2002"               # account that receives the annotated datasets (HF_TOKEN is its write token)
FILES_LIMIT = 3                         # 3 files x 20 answers = the first 60 prompts (same prompts for both models); None = all
HINDI_NOTE = True                       # add the short Hindi note after the paper's prompt (see top)
JUDGE = "openai/gpt-oss-120b:groq"
TEMPERATURE, MAX_TOKENS, MAX_SEARCHES = 0.0, 8192, 15    # repo: annotate.py / run.py
EXA_MCP = "https://mcp.exa.ai/mcp?tools=web_search_exa"   # Exa's hosted MCP server, keyless with a free rate limit
WORKERS = 4                             # answers in parallel; lower it if "Search failed" appears
COMPARE = ["longfact-hi-aya-expanse-8b-q8-0", "longfact-hi-llama3.1-8b-instruct-q8-0"]   # last cell

In [ ]:
# ---------------- the repo's annotation code ----------------
import ast, importlib.util, json, logging, re, time
from typing import List
sys.path.insert(0, "repo")
from utils.parsing import parse_and_validate_json
import utils.string_utils as string_utils
from utils.string_utils import try_matching_span_in_text
from rouge_score import rouge_scorer

spec = importlib.util.spec_from_file_location("data_models", "repo/annotation_pipeline/data_models.py")
data_models = importlib.util.module_from_spec(spec); spec.loader.exec_module(data_models)
AnnotatedSpan = data_models.AnnotatedSpan

# format_prompt and assign_span_positions, taken verbatim from annotate.py (importing the file itself would need Anthropic's safety-tooling)
repo_fns = {"List": List, "AnnotatedSpan": AnnotatedSpan, "try_matching_span_in_text": try_matching_span_in_text,
            "logger": logging.getLogger("annotate")}
for node in ast.parse(open("repo/annotation_pipeline/annotate.py", encoding="utf-8").read()).body:
    if isinstance(node, ast.FunctionDef) and node.name in ("format_prompt", "assign_span_positions"):
        exec(compile(ast.Module([node], []), "annotate.py", "exec"), repo_fns)
format_prompt, assign_span_positions = repo_fns["format_prompt"], repo_fns["assign_span_positions"]
PROMPT = open("repo/annotation_pipeline/entity_annotation.prompt", encoding="utf-8").read().strip()   # as in annotate.py

class UnicodeTokenizer:          # ROUGE's default tokenizer deletes non-Latin text; split on spaces/punctuation instead
    def tokenize(self, text):
        return [t for t in re.split(r"[\s.,;:!?\"'()\[\]\-‐‑‒–—―।]+", text.lower()) if t]
string_utils.ROUGE_SCORER = rouge_scorer.RougeScorer(["rougeL"], tokenizer=UnicodeTokenizer())
print(PROMPT[:200], "...")

In [ ]:
# ---------------- tokens, judge, web search ----------------
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass
from ddgs import DDGS
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError
from openai import OpenAI

try:                                    # Kaggle secret, else environment variable, else ask
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")   # shreshtha2002: write + "Make calls to Inference Providers"
api = HfApi(token=HF_TOKEN)
me = api.whoami()
assert HF_USER in [me["name"]] + [o["name"] for o in me.get("orgs", [])], f"HF_TOKEN belongs to '{me['name']}', not '{HF_USER}'"
client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN, max_retries=6, timeout=600)
log = lambda m: print(time.strftime("%H:%M:%S"), m, flush=True)

def exa_mcp(query):
    # minimal MCP (streamable HTTP) client: initialize -> initialized -> tools/call web_search_exa
    H = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream",
         "User-Agent": "Mozilla/5.0 (hallucination-annotator)"}
    def rpc(body, sid=None):
        req = urllib.request.Request(EXA_MCP, data=json.dumps(body).encode(), headers={**H, **({"Mcp-Session-Id": sid} if sid else {})})
        with urllib.request.urlopen(req, timeout=60) as r:
            sid = r.headers.get("Mcp-Session-Id") or sid
            txt = r.read().decode("utf-8")
        data = [l[5:].strip() for l in txt.splitlines() if l.startswith("data:")]
        msg = json.loads(data[-1]) if data else (json.loads(txt) if txt.strip() else {})
        if "error" in msg:
            raise RuntimeError(msg["error"])
        return sid, msg
    sid, _ = rpc({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                  "params": {"protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "annotator", "version": "0.1"}}})
    rpc({"jsonrpc": "2.0", "method": "notifications/initialized"}, sid)
    _, msg = rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                  "params": {"name": "web_search_exa", "arguments": {"query": query, "numResults": 5}}}, sid)
    return "\n\n".join(c.get("text", "") for c in msg["result"]["content"])

EXA_COOLDOWN = {"until": 0}
def web_search(query):            # Exa MCP when it isn't rate-limiting us; otherwise DuckDuckGo
    if time.time() > EXA_COOLDOWN["until"]:
        try:
            return exa_mcp(query)[:2500]          # keeps the judge's context (and your credits) small
        except Exception as e:
            EXA_COOLDOWN["until"] = time.time() + 600
            log(f"  Exa MCP failed ({str(e)[:80]}); DuckDuckGo only for the next 10 min")
    for attempt in range(3):
        try:
            hits = DDGS().text(query, max_results=5)
            return "\n\n".join(f"{h.get('title')}\n{h.get('href')}\n{h.get('body')}" for h in hits) or "No results."
        except Exception:
            time.sleep(3 * 2 ** attempt)
    log(f"  SEARCH FAILED: {query[:60]}")
    return "Search failed."

SEARCH = {"name": "web_search", "description": "Search the web. Returns the top results (title, URL, snippet).",
          "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}

def judge(user_prompt):
    """The repo's prompt as one user message; the judge may call web_search up to MAX_SEARCHES times."""
    n, found = 0, []
    tools = [{"type": "function", "function": SEARCH}]
    msgs = [{"role": "user", "content": user_prompt}]
    for _ in range(MAX_SEARCHES + 5):            # tools stay on (Groq rejects a tool call when tools are removed)
        m = client.chat.completions.create(model=JUDGE, messages=msgs, tools=tools, temperature=TEMPERATURE,
                                           max_tokens=MAX_TOKENS).choices[0].message
        if not m.tool_calls:
            return m.content or ""
        msgs.append({"role": "assistant", "content": m.content or "", "tool_calls": [t.model_dump() for t in m.tool_calls]})
        for t in m.tool_calls:
            n += 1
            query = json.loads(t.function.arguments or "{}").get("query", "")
            result = web_search(query) if n <= MAX_SEARCHES else "Search limit reached. Give your final answer now."
            if n <= MAX_SEARCHES:
                found.append(f"[search: {query}]\n{result}")
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": result})
    # rare: it kept searching past the cap -> one last call without tools, with the results it already gathered
    final = (user_prompt + "\n\nWeb search results already gathered:\n\n" + "\n\n".join(found)
             + "\n\nNo more searches are possible. Return the JSON array now.")
    return client.chat.completions.create(model=JUDGE, messages=[{"role": "user", "content": final}],
                                          temperature=TEMPERATURE, max_tokens=MAX_TOKENS).choices[0].message.content or ""

HINDI = ("\n\nNote: the instruction and the completion are in Hindi. Copy every \"span\" exactly as it appears in the completion, "
         "in Devanagari script, without translating or transliterating it. You may search in English or Hindi. "
         "Write the verification_note in English.")
ANNOTATOR = f"hf/{JUDGE}+exa-mcp/ddgs" + ("+hindi-note" if HINDI_NOTE else "")

def annotate(row):
    """Same steps as the repo's annotate_completion (instruction = conversation[-2], completion = conversation[-1])."""
    instruction, completion = row["conversation"][-2]["content"], row["conversation"][-1]["content"]
    prompt = format_prompt(instruction, completion, PROMPT) + (HINDI if HINDI_NOTE else "")
    for attempt in range(3):
        try:
            spans = parse_and_validate_json(judge(prompt), List[AnnotatedSpan], allow_partial=True)
            spans = assign_span_positions(spans, completion)
            log(f"  done {row['prompt_id']}: {len(spans)} entities")
            return {**row, "annotations": [s.model_dump() for s in spans], "annotator": ANNOTATOR}
        except Exception as e:
            log(f"  attempt {attempt + 1} failed for {row['prompt_id']}: {str(e)[:300]}")
    return {**row, "annotations": None, "annotator": None}   # failed; left unlabelled

print("search test:", web_search("टाइलेनॉल हत्याएं 1982 शिकागो")[:300])

In [ ]:
# ---------------- annotate each file, saving after every answer (Run again to resume) ----------------
import threading
SAVE_EVERY = 5          # also upload progress to Hugging Face every 5 finished answers

def download(repo_id, filename):
    # plain urllib download (avoids an old-brotli crash inside huggingface_hub); public datasets need no token
    path = os.path.join("downloads", repo_id.replace("/", "__"), filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with urllib.request.urlopen(f"https://huggingface.co/datasets/{repo_id}/resolve/main/{filename}", timeout=120) as r, open(path, "wb") as fh:
        fh.write(r.read())
    return path

src, dst = f"{SRC_USER}/{MODEL_REPO}", f"{HF_USER}/{MODEL_REPO}-annotated"
api.create_repo(dst, repo_type="dataset", private=False, exist_ok=True)
done = set(api.list_repo_files(dst, repo_type="dataset"))

# every finished answer is appended here at once; on a re-run these answers are not redone
PROGRESS = os.path.join("work", MODEL_REPO, "progress.jsonl")
os.makedirs(os.path.dirname(PROGRESS), exist_ok=True)
if not os.path.exists(PROGRESS) and "progress/progress.jsonl" in done:      # resume from the Hub copy (e.g. another machine)
    with open(PROGRESS, "w", encoding="utf-8") as fh:
        fh.write(open(download(dst, "progress/progress.jsonl"), encoding="utf-8").read())
saved = {}
if os.path.exists(PROGRESS):
    for l in open(PROGRESS, encoding="utf-8"):
        r = json.loads(l); saved[r["prompt_id"]] = r
lock, since_upload = threading.Lock(), [0]

def push_progress():
    api.upload_file(path_or_fileobj=PROGRESS, path_in_repo="progress/progress.jsonl", repo_id=dst, repo_type="dataset",
                    commit_message=f"progress: {len(saved)} answers")

def annotate_and_save(row):
    key = row["prompt_id"]
    if key in saved:
        return saved[key]
    r = annotate(row)
    if r["annotations"] is not None:               # failures are not saved, so they are retried on the next run
        with lock:
            saved[key] = r
            with open(PROGRESS, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
            since_upload[0] += 1
            if since_upload[0] >= SAVE_EVERY:
                since_upload[0] = 0
                try:
                    push_progress()
                except Exception as e:
                    log(f"  progress upload failed (kept locally): {str(e)[:100]}")
    return r

files = sorted(f for f in api.list_repo_files(src, repo_type="dataset") if f.startswith("data/") and f.endswith(".jsonl"))
files = files[:FILES_LIMIT] if FILES_LIMIT else files
log(f"===== {src} -> https://huggingface.co/datasets/{dst}  ({len(files)} files, {len(done & set(files))} done, {len(saved)} answers saved)")
for f in files:
    if f in done:
        continue
    t = time.time()
    rows = [json.loads(l) for l in open(download(src, f), encoding="utf-8")]
    with ThreadPoolExecutor(WORKERS) as pool:
        out = list(pool.map(annotate_and_save, rows))
    local = os.path.join("work", MODEL_REPO, f)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    with open(local, "w", encoding="utf-8") as fh:
        fh.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in out)
    api.upload_file(path_or_fileobj=local, path_in_repo=f, repo_id=dst, repo_type="dataset", commit_message=f"annotate {f}")
    push_progress()
    labels = [a["label"] for r in out for a in r["annotations"] or [] if a["index"] is not None]
    log(f"  {f}: {len(out)} answers ({sum(r['annotations'] is None for r in out)} failed), {len(labels)} entities: "
        f"{labels.count('Supported')} S / {labels.count('Not Supported')} NS / {labels.count('Insufficient Information')} II "
        f"in {time.time() - t:.0f}s -> saved {local}")
log("All done.")

In [ ]:
# ---------------- Hindi hallucination rate per model, on the prompts every model has (from everything saved so far) ----------------
data = {}
for name in COMPARE:
    path = os.path.join("work", name, "progress.jsonl")
    if not os.path.exists(path):          # the other model ran elsewhere: use its Hub copy
        try:
            path = download(f"{HF_USER}/{name}-annotated", "progress/progress.jsonl")
        except Exception:
            print(f"skipping {name}: nothing saved yet"); continue
    data[name] = {r["prompt_id"]: r for r in map(json.loads, open(path, encoding="utf-8")) if r["annotations"] is not None}
common = sorted(set.intersection(*(set(d) for d in data.values()))) if data else []
print(f"{len(common)} prompts annotated for every model shown\n")
print(f"{'model':<42}{'answers':>8}{'entities':>9}{'NS %':>7}{'NS+II %':>9}{'w/ >=1 NS %':>13}{'0 entities %':>14}")
for name, d in data.items():
    spans = [[a["label"] for a in d[p]["annotations"] if a["label"] and a["index"] is not None] for p in common]
    flat = [l for s in spans for l in s]
    ns, ii = flat.count("Not Supported"), flat.count("Insufficient Information")
    print(f"{name:<42}{len(common):>8}{len(flat):>9}{100 * ns / max(1, len(flat)):>7.1f}{100 * (ns + ii) / max(1, len(flat)):>9.1f}"
          f"{100 * sum('Not Supported' in s for s in spans) / max(1, len(spans)):>13.1f}{100 * sum(not s for s in spans) / max(1, len(spans)):>14.1f}")
print("\n'0 entities' is mostly refusals / very short answers: they lower the hallucination rate, so report it alongside.")